In [3]:
from langchain.llms.openai import OpenAI
from langchain.chat_models import ChatOpenAI, ChatAnthropic

# temperature는 모델의 창의성을 결정하는 constructor. 0~1까지의 숫자를 입력할 수 있으며, 높을 수록 창의력이 높고 무작위.
chat = ChatOpenAI(
    temperature = 0.1
)
# The default model `text-davinci-003` has been deprecated.
llm = OpenAI(
    model="davinci-002"
)

In [ ]:
'''
.env 파일 안에 api_key가 먹지 않을 떄:

chat = ChatOpenAI(
    openai_api_key = ""
)
'''

In [ ]:
'''
# predict to Q&A

a = chat.predict("What am I gonna do when i'm sleepy")
b = llm.predict("What am I gonna do when i'm sleepy")

a,b
'''

In [ ]:
'''
# predict messages

# import message constructors
from langchain.schema import HumanMessage, AIMessage, SystemMessage

# custom prompts.
messages = [
    SystemMessage(
        content="You are Musicologist. Particularly expert in Electronic Dance Music."
    ),
    AIMessage(content="Ciao, mi chiamo Paolo."),
    HumanMessage(content="Hello, What's your name? What exactly is Nu-Disco? Is How sweet by NewJeans belong to this genre?"),
]

chat.predict_messages(messages)
'''

AIMessage(content='Hello! Nu-disco is a genre of dance music that blends elements of disco, funk, and electronic music. It often features a modern twist on the classic disco sound, with funky basslines, catchy melodies, and a more contemporary production style.\n\n"Sweet" by NewJeans is a track that falls under the nu-disco genre. It incorporates elements of disco and funk with a modern electronic production, making it a perfect example of the nu-disco sound. The track typically features a groovy bassline, funky guitar riffs, and catchy vocals that are characteristic of the genre.')

# Prompt Templates(#3.2)

- 모든 서비스들이 프롬프트 제작에 전념하고 있고, LangChain은 프롬프트 공유를 위한 커뮤니티를 만들고 있다.

In [ ]:
# PromptTemplate: create a template from just a string <-> ChatPromptTemplate: create a template from messages.
from langchain.prompts import PromptTemplate, ChatPromptTemplate

'''
# PromptTemplate

template = PromptTemplate.from_template("What is the distintive feature of {main_genre} compared to {genre_compared}")
# format {placeholders}
prompt = template.format(main_genre="Nu-Disco", genre_compared="Disco",)

chat.predict(prompt)

# ChatPromptTemplate

# replace [4] 'import message constructors'
template = ChatPromptTemplate.from_messages(
    [
    ("system", "You are Musicologist. Particularly expert in {genre}."),
    ("ai", "Ciao, mi chiamo {name}."),
    ("human", "Hello, What's your name? What exactly is {sub_genre}? Is {song_name} by {artist} belong to this genre?"),
    ]
)
prompt = template.format_messages(
    genre = "electronic",
    name = "Nico",
    sub_genre = "Nu-Disco",
    song_name = "How Sweet",
    artist = "NewJeans",
)

chat.predict_messages(prompt)
'''

AIMessage(content='Hello Nico! Nu-disco is a genre of dance music that blends elements of disco, funk, and electronic music. It often features modern production techniques and a contemporary sound while paying homage to the classic disco era.\n\n"Nu-disco" is characterized by its funky basslines, catchy melodies, and use of synthesizers and electronic beats. It aims to capture the essence of disco music while giving it a fresh, modern twist.\n\nAs for the track "How Sweet" by NewJeans, it could potentially fall under the nu-disco genre based on its production style and sound. However, without listening to the track myself, I can\'t provide a definitive answer. If you\'d like, I can help you analyze the song further to determine if it fits within the nu-disco genre.')

# Output Parser(#3.3)
## LLM Response into a List.

In [ ]:
# 기본 Output Parser를 확장
from langchain.schema import BaseOutputParser

class CommaOutputParser(BaseOutputParser):
    def parse(self, text):
        items = text.split(",")
        # return list형태로(each of item in items(item을 strip해서))
        return list(map(str.strip, items))

p = CommaOutputParser()
'''
p.parse("Hello,how, are,you")
'''

['Hello', 'how', 'are', 'you']

In [ ]:
# 만든 Output Parser 사용
template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a list generating machine. Everything you are asked will be answered with a comma separated list of max {max_items} in lowercase. Do NOT reply with anything else."),
        ("human", "{question}"),
    ]
)

prompt = template.format_messages(
    max_items = 5,
    question = "What type of exterior building materials are there?",
)

result = chat.predict_messages(prompt)
# get AIMessage
p.parse(result.content)

['brick',
 'wood',
 'stucco',
 'stone',
 'metal',
 'vinyl',
 'concrete',
 'glass',
 'aluminum']

# LangChain Expression Language(#3.3)
## [17] 대체
- format_messages
- chat.predict
- parser.parse

## only need
- Template
- Language Model
- Output Parser

In [19]:
# Create Chain: 하나의 체인으로 합쳐 순서대로 실행
chain = template | chat | CommaOutputParser()
chain.invoke(
    {
        "max_items":10,
        "question":"give me exterior building materials order by low expense",
    }
)

['concrete', 'vinyl siding', 'stucco', 'wood', 'brick']

In [ ]:
'''
# 체인 간 결합도 가능
chain_one = template | chat | CommaOutputParser()
chain_two = template_2 | chat | OutputParser_2

all_combine = chain_one | chain_two | OutputParser_3
'''